In [5]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from ipywidgets import interact, IntSlider
import warnings
warnings.filterwarnings('ignore')

In [6]:
regional_prices = {
    'Dar Es Salaam':        2288.12,
    'Mainland Other Urban': 1946.76,
    'Mainland Rural':       1936.82,
    'Zanzibar':             2459.13,
}
AVG_PRICE = np.mean(list(regional_prices.values()))


PROTEIN_PER_KG = 89.0
KCAL_PER_KG    = 1270.0


PROTEIN_CURRENT  = 42.0;  PROTEIN_REQUIRED = 56.0
KCAL_CURRENT     = 1900.0; KCAL_REQUIRED   = 2100.0


PROTEIN_GOAL = PROTEIN_CURRENT + 0.5 * (PROTEIN_REQUIRED - PROTEIN_CURRENT) 
KCAL_GOAL    = KCAL_CURRENT    + 0.5 * (KCAL_REQUIRED    - KCAL_CURRENT)    


ELASTICITY          = -0.7
BASE_CONSUMPTION_KG = 0.060 

print(f"National average: {AVG_PRICE:,.0f} TZS/kg")
print(f"Protein goal: {PROTEIN_GOAL:.1f} g/day | Calorie goal: {KCAL_GOAL:.0f} kcal/day")

National average: 2,158 TZS/kg
Protein goal: 49.0 g/day | Calorie goal: 2000 kcal/day


In [7]:
def get_consumption(subsidized_price):
    pct_change = (subsidized_price - AVG_PRICE) / AVG_PRICE
    return max(0.001, BASE_CONSUMPTION_KG * (1 + ELASTICITY * pct_change))

def get_intakes(subsidized_price):
    cons_kg = get_consumption(subsidized_price)
    protein = min(PROTEIN_REQUIRED + 5,   PROTEIN_CURRENT + cons_kg * PROTEIN_PER_KG)
    kcal    = min(KCAL_REQUIRED    + 100, KCAL_CURRENT    + cons_kg * KCAL_PER_KG)
    return protein, kcal, cons_kg

In [8]:
def plot_subsidy(subsidized_price):
    protein, kcal, cons_kg = get_intakes(subsidized_price)
    subsidy_per_kg = max(0, AVG_PRICE - subsidized_price)
    pct_discount   = (subsidy_per_kg / AVG_PRICE) * 100
    protein_met = protein >= PROTEIN_GOAL
    kcal_met    = kcal    >= KCAL_GOAL
    both_met    = protein_met and kcal_met

    C_BASELINE='#B4B2A9'; C_SUBSIDY='#1D9E75'; C_GOAL='#D85A30'; C_REQ='#888780'
    C_PROTEIN='#3266ad';  C_KCAL='#E28A2B'

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))
    fig.patch.set_facecolor('#f9f9f9')

   
    ax1.set_facecolor('white')
    x = np.arange(2); w = 0.3
    ax1.bar(x-w/2, [PROTEIN_CURRENT, KCAL_CURRENT/10], width=w, color=C_BASELINE, label='Current baseline', zorder=3)
    ax1.bar(x+w/2, [protein, kcal/10],                 width=w, color=C_SUBSIDY,  label='With subsidy',     zorder=3)
    for i,(g,r) in enumerate(zip([PROTEIN_GOAL, KCAL_GOAL/10],[PROTEIN_REQUIRED, KCAL_REQUIRED/10])):
        ax1.hlines(g, x[i]-.38, x[i]+.38, colors=C_GOAL, linewidths=2.5, linestyles='--', zorder=4)
        ax1.hlines(r, x[i]-.38, x[i]+.38, colors=C_REQ,  linewidths=1.5, linestyles=':',  zorder=4)
    for i,(b,s) in enumerate(zip([PROTEIN_CURRENT, KCAL_CURRENT/10],[protein, kcal/10])):
        ax1.text(x[i]-w/2, b+.5, f'{b:.1f}', ha='center', va='bottom', fontsize=9, color='#555')
        met = protein_met if i==0 else kcal_met
        ax1.text(x[i]+w/2, s+.5, f'{s:.1f}', ha='center', va='bottom', fontsize=9,
                 color=C_SUBSIDY if met else C_GOAL, fontweight='bold')
    ax1.set_xticks(x); ax1.set_xticklabels(['Protein (g/day)', 'Calories (kcal ÷ 10)'], fontsize=11)
    ax1.set_ylim(0, 230); ax1.set_title('Protein & Calorie Intake vs. Goals', fontsize=12, fontweight='bold')
    ax1.grid(axis='y', alpha=0.3); ax1.spines[['top','right']].set_visible(False)
    ax1.legend(handles=[
        mpatches.Patch(color=C_BASELINE, label='Current baseline'),
        mpatches.Patch(color=C_SUBSIDY,  label='With subsidy'),
        plt.Line2D([0],[0], color=C_GOAL, lw=2,   linestyle='--', label='50% deficit goal'),
        plt.Line2D([0],[0], color=C_REQ,  lw=1.5, linestyle=':',  label='Full requirement'),
    ], fontsize=9)

    
    ax2.set_facecolor('white')
    prices = np.linspace(400, int(AVG_PRICE), 300)
    ax2.plot(prices, [get_intakes(p)[0]      for p in prices], color=C_PROTEIN, lw=2.5, label='Protein (g/day)')
    ax2.plot(prices, [get_intakes(p)[1]/10   for p in prices], color=C_KCAL,    lw=2.5, linestyle='--', label='Calories (kcal ÷ 10)')
    ax2.axhline(PROTEIN_GOAL,    color=C_GOAL, lw=1.5, linestyle='--', alpha=.7, label=f'Protein goal ({PROTEIN_GOAL:.1f}g)')
    ax2.axhline(KCAL_GOAL/10,   color=C_KCAL, lw=1.5, linestyle=':',  alpha=.7, label=f'Calorie goal ({KCAL_GOAL:.0f} kcal)')
    ax2.axvline(subsidized_price, color='#333', lw=1.5, linestyle='-', alpha=.5, label=f'Slider: {subsidized_price:,} TZS/kg')
    ax2.set_xlim(400, AVG_PRICE); ax2.set_ylim(40, 80); ax2.invert_xaxis()
    ax2.set_xlabel('Subsidized price (TZS/kg)', fontsize=10)
    ax2.set_title('Response Curve: Price → Nutrition', fontsize=12, fontweight='bold')
    ax2.grid(alpha=0.3); ax2.spines[['top','right']].set_visible(False); ax2.legend(fontsize=9)

    fig.suptitle(
        f"{'✓ Both goals met!' if both_met else '⚠ Goals not yet met'}   |   "
        f"Price: {subsidized_price:,} TZS/kg   |   Subsidy: {subsidy_per_kg:,.0f} TZS/kg ({pct_discount:.0f}% off)   |   "
        f"Bean consumption: {cons_kg*1000:.1f} g/person/day",
        fontsize=11, color='#1D9E75' if both_met else '#D85A30', y=1.02, fontweight='bold'
    )
    plt.tight_layout(); plt.show()

   
    print(f"\n{'Region':<26} {'Market Price':>14} {'Subsidized':>12} {'Subsidy/kg':>12}")
    print('-'*67)
    for region, price in regional_prices.items():
        print(f"{region:<26} {price:>12,.0f}   {max(400,price-subsidy_per_kg):>10,.0f}   {subsidy_per_kg:>10,.0f}")
    print(f"{'National Average':<26} {AVG_PRICE:>12,.0f}   {subsidized_price:>10,}   {subsidy_per_kg:>10,.0f}")

interact(plot_subsidy, subsidized_price=IntSlider(
    value=int(AVG_PRICE), min=400, max=int(AVG_PRICE), step=10,
    description='Price (TZS/kg):', style={'description_width':'initial'}, layout={'width':'600px'}
));

interactive(children=(IntSlider(value=2157, description='Price (TZS/kg):', layout=Layout(width='600px'), max=2…